## 01 SARSA

In [1]:
import time
import numpy as np
import gymnasium as gym 

import Toy_Envs.gridworld as gv

SARSA: State-Action-Reward-State-Action

Step-1 TD algorithm is to solve **Bellman Equation** by using stochastic approx, 

$$
\begin{aligned}
q_{t+1}(s_t, a_t) &= q_t(s_t, a_t) - \alpha_t(s_t, a_t)[q_t(s_t, a_t) - [r_{t+1} + \gamma q_t(s_{t+1}, a_{t+1})]] \\ 
q_{t+1}(s, a) &= q_t(s, a), \forall (s, a) \neq (s_t, a_t)
\end{aligned}
$$

$$
\begin{aligned}
\pi_{t+1}(a | s_t) &= 1 - \frac{\epsilon}{|\mathcal{A}(s)|}(|\mathcal{A}(s)| - 1) \text{ if } a = \argmax_a q_{t+1}(s_t, a) \\ 
\pi_{t+1}(a | s_t) &= \frac{\epsilon}{|\mathcal{A}(s)|}, \text{ otherwise}
\end{aligned}
$$

In [2]:
class Sarsa_Agent():
  """action is decretized as {0, 1, 2, 3} in wrapper, so action is numbered as number"""
  
  def __init__( self,
                obs_dim: int,
                action_dim: int,
                epsilon: float = 0.1,
                lr: float = 0.1,
                gamma: float = 0.9
                ) -> None:
    
    self.obs_dim = obs_dim
    self.action_dim = action_dim
    self.Q = np.zeros((self.obs_dim, self.action_dim))
    
    self.epsilon = epsilon
    self.lr = lr
    self.gamma = gamma
    
  def get_greedy_action(self, obs: int) -> int:
    # action of determine policy by policy improvement
    Q_list = self.Q[obs,:]
    # action = np.argmax(Q_list) # constant action
    action = np.random.choice(np.flatnonzero(Q_list==Q_list.max()))
    return action
  
  def get_action(self, obs: int) -> int:
    # epsilon-greedy policy
    if np.random.uniform(0, 1) < self.epsilon:
      action = np.random.choice(self.action_dim)
    else:
      # use improved policy
      action = self.get_greedy_action(obs)
      
    return action
  
  def policy_evaluation(self,
                        obs: int,
                        action: float,
                        reward: float,
                        next_obs: int,
                        next_action: int,
                        done: bool) -> None:
    current_Q = self.Q[obs, action]
    # if terminated is True, there'll be no next_state and next_action
    # thus reward is just target_Q
    TD_target = reward + (1 - float(done)) * self.gamma * self.Q[next_obs, next_action]
    self.Q[obs, action] -= self.lr * (current_Q - TD_target)

In [3]:
class TrainManager():
  
  def __init__( self,
                env: gym.Env,
                episode_num: int=1000,
                lr:float=0.1,
                gamma:float=0.9,
                epsilon:float=0.1) -> None:
    self.env = env
    self.episode_num = episode_num
    obs_dim = env.observation_space.n 
    action_dim = env.action_space.n 
    self.agent = Sarsa_Agent(
      obs_dim = obs_dim,
      action_dim=action_dim,
      epsilon=epsilon,
      lr=lr,
      gamma=gamma
    )
    
  def train_episode(self, is_render:bool=False)->float:
    total_reward = 0 # record ttl reward in 1 epi
    obs, _ = self.env.reset() # reset env and get init state
    while True:
      # get action using learnt epsilon-greedy policy
      action = self.agent.get_action(obs) 
      # take act and get next_state, r, done, info
      next_obs, reward, terminated, truncated, _ = self.env.step(action) 
      """In Gymnasium or Gym v0.26, done is True when terminated or truncated,
      In Gym early version, there is done but not terminated, truncated here.",
      You should modify the code according to the version of Gym you use.",
      """
      done = terminated or truncated
      total_reward += reward
      # For Sarsa, we Need obtain a' using cur policy
      next_action = self.agent.get_action(next_obs)
      # using data to do policy eval
      self.agent.policy_evaluation(obs, action, reward, next_obs, next_action, done)
      # update state and action
      obs = next_obs
      if is_render:
        self.env.render() # You can find the winodw in taskbar
        time.sleep(0.1) # set a speed for vis
      if done:
        break
    
    return total_reward
  
  def test_episode(self) -> float:
    total_reward = 0 # record total rwd in 1 epi
    obs, _ = self.env.reset() # reset env and get init state
    while True:
      action = self.agent.get_greedy_action(obs) # get action using learnt greedy policy
      next_obs, reward, terminated, truncated, _ = self.env.step(action) # take action and get next_state
      done = terminated or truncated
      obs = next_obs
      total_reward += reward
      self.env.render()
      time.sleep(0.1)
      if done:
        break

    return total_reward
  
  def train(self) -> None:
    is_render = False
    for e in range(self.episode_num):
      episode_reward = self.train_episode(is_render)
      print('Episode %s: Total Reward = %.2f'%(e, episode_reward))
      
      if e % 50 == 0:
        is_render = True
      else:
        is_render = False
        
    test_reward = self.test_episode()
    print('Test Total Reward = %.2f' % (test_reward))
    return

In [ ]:
env = gym.make('CliffWalking-v0')
env = gv.CliffWalkingWapper(env)

"""Here is another game"""
env = gym.make('FrozenLake-v1', is_slippery=False)
env = gv.FrozenLakeWrapper(env)

Manager = TrainManager( env=env,
                        episode_num=500,
                        lr=0.1,
                        gamma=0.9,
                        epsilon=0.1)
Manager.train()

Episode 0: Total Reward = 0.00
Episode 1: Total Reward = 0.00
Episode 2: Total Reward = 0.00
Episode 3: Total Reward = 0.00
Episode 4: Total Reward = 0.00
Episode 5: Total Reward = 0.00
Episode 6: Total Reward = 0.00
Episode 7: Total Reward = 0.00
Episode 8: Total Reward = 0.00
Episode 9: Total Reward = 0.00
Episode 10: Total Reward = 0.00
Episode 11: Total Reward = 0.00
Episode 12: Total Reward = 0.00
Episode 13: Total Reward = 0.00
Episode 14: Total Reward = 0.00
Episode 15: Total Reward = 0.00
Episode 16: Total Reward = 0.00
Episode 17: Total Reward = 0.00
Episode 18: Total Reward = 0.00
Episode 19: Total Reward = 0.00
Episode 20: Total Reward = 0.00
Episode 21: Total Reward = 0.00
Episode 22: Total Reward = 0.00
Episode 23: Total Reward = 0.00
Episode 24: Total Reward = 0.00
Episode 25: Total Reward = 0.00
Episode 26: Total Reward = 0.00
Episode 27: Total Reward = 0.00
Episode 28: Total Reward = 0.00
Episode 29: Total Reward = 0.00
Episode 30: Total Reward = 0.00
Episode 31: Total 

: 